# kodoom: current execution

Active stage: **free-text-gate** (project-plan.md: 1.2 step 1 (v17)).
Run all performs a 3-record/4-case GPU preflight, then the 40-record helmo and
20-case typed-decisions training gate, using **gemma3-4b-bf16**.
It ends at **awaiting-review**, not at full translation.

Before starting: create a private Kaggle notebook, enable Internet and a GPU,
accept `google/gemma-3-4b-it` on Hugging Face, and enable the `HF_TOKEN` secret.
See `docs/execution-workflow.md` for saving/restoring and provider settings.


In [ ]:
PROVIDER = "kaggle"  # kaggle, colab, generic
PROFILE = "kaggle"  # colab on Colab; custom TOML path for generic
CODE_REVISION = 'cfd6e8a5926ea2f7f8e533182047a2ce9fccf1a8'
CHECKOUT = "/tmp/kodoom-code"  # use a new path when changing code revisions
RESTORE_PREFLIGHT = ""  # optional attached preflight ZIP file
RESTORE_GATE = ""  # optional attached gate ZIP file


## Bootstrap the pinned code


In [ ]:
import os
import types
import urllib.parse
import urllib.request

if PROVIDER == "kaggle":
    from kaggle_secrets import UserSecretsClient
    try:
        os.environ["GITHUB_TOKEN"] = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        pass  # optional for a public repository
elif PROVIDER == "colab":
    from google.colab import userdata
    try:
        os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")
    except Exception:
        pass
url = ("https://api.github.com/repos/Farahani1/kodum/contents/src/kodoom/runtime.py"
       + "?ref=" + urllib.parse.quote(CODE_REVISION, safe=""))
headers = {"Accept": "application/vnd.github.raw+json"}
if os.environ.get("GITHUB_TOKEN"):
    headers["Authorization"] = "Bearer " + os.environ["GITHUB_TOKEN"]
with urllib.request.urlopen(urllib.request.Request(url, headers=headers)) as response:
    source = response.read().decode("utf-8")
runtime = types.ModuleType("kodoom_bootstrap")
exec(compile(source, "runtime.py", "exec"), runtime.__dict__)
checkout = runtime.bootstrap(PROVIDER, CODE_REVISION, CHECKOUT)


## Restore selected private artifacts


In [ ]:
from kodoom.workflow import restore_stage

if RESTORE_PREFLIGHT:
    restore_stage(PROVIDER, PROFILE, "preflight", RESTORE_PREFLIGHT)
if RESTORE_GATE:
    restore_stage(PROVIDER, PROFILE, "gate", RESTORE_GATE)


## GPU preflight


In [ ]:
from kodoom.workflow import run_current

preflight = run_current(provider=PROVIDER, profile=PROFILE,
                        mode="preflight")


## Free-text gate


In [ ]:
gate = run_current(provider=PROVIDER, profile=PROFILE,
                   mode="gate",
                   model_revision=preflight["model_revision"])


## Private output and review

At most 1 meaning change in 40 helmo records; at most 10-15% needing edits. Human review required; do not start full translation.

Save a private version with outputs, then attach that output in a new session and restore its ZIP to verify persistence. Source-only saves do not prove durability. The live session may lose all progress since its last saved output.


In [ ]:
print("Gate status:", gate["status"])
print("Save this private output bundle:", gate["bundle"])
print("Review sheets are in the bundle. Stop here for the main-plan review decision.")
